# Jour 2 · Chapitre 1 — Scénario : préparer le rendez-vous d'un client multi-équipé
## Formation Beobank · SAS → Python · Cahier participant

**L'histoire.** Demain à 10h, le client n° `2900000004654` a rendez-vous avec la directrice
d'agence. C'est le client qui détient le plus de comptes chez Beobank. Elle vous écrit :
*"Je veux une fiche claire : combien de comptes, dans quelles devises, quels statuts, et
l'état de ses soldes. Prépare-moi ça avec des listes et des dictionnaires, je veux pouvoir
réutiliser la structure pour d'autres clients."*

**Comment travailler.** Remplacez chaque `______`. Les commentaires indiquent la notion à
mobiliser ; à vous de retrouver la syntaxe exacte dans la matière.

📎 **Matière à relire :** `Cours_Listes_Dicts_Tuples.ipynb` (même dossier) — Partie A (listes),
Partie B (dictionnaires), Partie C (tuples), sets et copies.

## Préparation (déjà écrite, exécutez-la)

In [ ]:
# ── Lecture des fichiers en listes de dictionnaires ───────────────────────
import csv
import copy

def lire(nom_fichier):
    """Lit un CSV Beobank et renvoie une liste de dictionnaires (une ligne = un dict)."""
    with open("../../data/" + nom_fichier, encoding="utf-8") as f:
        return list(csv.DictReader(f, delimiter=";"))

contrats = lire("CTR.csv")          # 200 contrats
liens    = lire("TIE_X_CTR.csv")    # qui possède quel compte
CLIENT   = "2900000004654"          # le client du rendez-vous (NUM_TIE, lu comme du texte)
print(len(contrats), "contrats,", len(liens), "liens client-compte")

---
## Mission 1 — La liste de ses comptes

*"Commence par me lister ses numéros de compte."*

In [ ]:
# ── Listes : création, longueur, accès, tranche ───────────────────────────
# Quel opérateur teste l'égalité de deux valeurs ?
comptes_client = [l["IDT_AC"] for l in liens if l["NUM_TIE"] ______ CLIENT]
# Quelle fonction donne le nombre d'éléments d'une liste ?
print("Nombre de comptes :", ______(comptes_client))          # → 46
# Quel index pour le PREMIER élément ? Et pour le DERNIER sans connaître la longueur ?
print("Premier :", comptes_client[______], "| Dernier :", comptes_client[______])
# Quelle tranche (slice) donne les 3 premiers éléments ?
print("Aperçu :", comptes_client[______])

---
## Mission 2 — Retrouver un contrat en une fraction de seconde

*"Chercher dans une liste de 200 contrats à chaque fois, c'est lent. Fais un index."*

In [ ]:
# ── Dictionnaire en compréhension = un index ──────────────────────────────
# Quelle valeur doit servir de CLÉ pour retrouver un contrat par son numéro de compte ?
index_contrats = {______: c for c in contrats}
fiche = index_contrats[comptes_client[0]]
# Quelle méthode lit une clé sans planter si elle n'existe pas ?
print("Devise :", fiche.______("COD_DEV"), "| Solde :", fiche.get("SLD_CTR"))
# Quelle méthode liste toutes les clés d'un dictionnaire ?
print(list(fiche.______()))

---
## Mission 3 — Ses soldes connus

*"Je veux tous ses soldes, du plus bas au plus haut. Ignore ceux qui sont inconnus."*

Dans le fichier, un solde inconnu est écrit `.`.

In [ ]:
# ── Construire et trier une liste ─────────────────────────────────────────
soldes = []
for idt in comptes_client:
    s = index_contrats[idt]["SLD_CTR"]
    # Quelle valeur représente un solde inconnu dans les exports SAS ?
    if s != "______":
        # Quelle méthode ajoute un élément à la fin de la liste ?
        soldes.______(float(s))
# Quelle méthode trie la liste sur place (du plus petit au plus grand) ?
soldes.______()
print(len(soldes), "soldes connus :", soldes[:3], "...")      # → 22 soldes connus

---
## Mission 4 — Des fiches compte non modifiables

*"Pour l'audit, chaque compte doit être une fiche figée : numéro, devise, statut."*

In [ ]:
# ── Tuples et déballage (unpacking) ───────────────────────────────────────
fiches = [(idt, index_contrats[idt]["COD_DEV"], index_contrats[idt]["COD_ECV_CTR"])
          for idt in comptes_client]                          # liste de tuples
premier = fiches[0]
try:
    premier[1] = "USD"                                         # on tente de modifier un tuple...
# Quelle erreur Python lève une tentative de modification d'un tuple ?
except ______ as erreur:
    print("Impossible :", erreur)

# Déballage : une variable par élément du tuple, dans le même ordre que la fiche
for ______ in fiches[:3]:
    print(idt, devise, statut)

---
## Mission 5 — Statuts et devises

*"Combien de comptes clôturés, combien de résiliés ? Et il a des comptes en devises étrangères ?"*

In [ ]:
# ── Comptage avec un dict + ensemble (set) ────────────────────────────────
libelles = {"4": "Clôturé", "6": "Résilié"}
compteur_statuts = {}
for _, _, statut in fiches:
    libelle = libelles[statut]
    # Quelle méthode crée la clé avec une valeur par défaut si elle n'existe pas, puis la renvoie ?
    compteur_statuts.______(libelle, 0)
    compteur_statuts[libelle] += 1
print(compteur_statuts)

# Un set ne garde chaque valeur qu'une fois : quelle fonction transforme une liste en set ?
devises = ______(devise for _, devise, _ in fiches)
print("Devises :", sorted(devises))                           # → ['EUR', 'USD']
# Tous ses comptes existent-ils dans CTR ? Quel opérateur de set donne « ce qui manque » ?
print("Comptes introuvables :", set(comptes_client) ______ set(index_contrats))   # → set()

---
## Mission 6 — La fiche client complète

*"Regroupe tout dans une seule structure que je peux envoyer à l'outil CRM."*

In [ ]:
# ── Dictionnaire imbriqué ─────────────────────────────────────────────────
fiche_client = {
    "num_tie": CLIENT,
    "nb_comptes": len(comptes_client),
    "devises": sorted(devises),
    "statuts": compteur_statuts,
    # Quelles fonctions donnent le plus petit et le plus grand élément d'une liste ?
    "soldes": {"min": ______(soldes), "max": ______(soldes), "nb_decouverts": sum(1 for s in soldes if s < 0)},
}
# Comment lire la valeur "max" À L'INTÉRIEUR du sous-dictionnaire "soldes" ?
print("Meilleur solde :", fiche_client["soldes"]______)

---
## Mission 7 — Le piège de la copie

*"Avant de simuler un scénario, garde une copie intacte de la fiche."*

In [ ]:
# ── Copie superficielle vs copie profonde ─────────────────────────────────
# Le sous-dictionnaire "soldes" est partagé par une copie simple : laquelle copie AUSSI les sous-objets ?
sauvegarde = copy.______(fiche_client)
fiche_client["soldes"]["nb_decouverts"] = 0                   # simulation : tous les découverts régularisés
print("Simulation :", fiche_client["soldes"]["nb_decouverts"],
      "| Sauvegarde intacte :", sauvegarde["soldes"]["nb_decouverts"])     # → 0 | 3

---
## 🏁 Mission finale — L'impression pour la directrice

Format attendu :
```
FICHE CLIENT 2900000004654
  Comptes        : 46  (EUR, USD)
  Clôturé        : ..
  Résilié        : ..
  Soldes connus  : 22  | min ... | max ...
  Découverts     : 3
```

In [ ]:
# ── Rapport final ─────────────────────────────────────────────────────────
f = sauvegarde                                                # on imprime la version d'origine
print(f"FICHE CLIENT {f['num_tie']}")
print(f"  Comptes        : {f['nb_comptes']}  ({', '.join(f['devises'])})")
# Quelle méthode de dict donne les paires (clé, valeur) ? Quel mot-clé crée une petite fonction sans nom ?
for libelle, nombre in sorted(f["statuts"].______(), key=______ paire: paire[1], reverse=True):
    print(f"  {libelle:<14} : {nombre}")
print(f"  Soldes connus  : {len(soldes)}  | min {f['soldes']['min']:,.2f} | max {f['soldes']['max']:,.2f}")
print(f"  Découverts     : {f['soldes']['nb_decouverts']}")

✍️ **Ce que ça veut dire pour le rendez-vous :** en 2 phrases, quel message la directrice doit-elle
préparer (nombre de comptes, découverts, devise étrangère) ?